In [1]:
using Plots, JLD2
include("analysis_tools.jl");

In [2]:
# setting up the problem

# parameters
nx=100; ny=100     # number of spatial points
Lx=10.0; Ly=10.0     # length of the domain
Δx=Lx/nx; Δy=Ly/ny    # spatial step
ν=0.1    # viscosity
kx=2π/Lx; ky=2π/Ly   # wave number for initial condition
u_mean=1 # mean velocity
u_amplitude=0.5 # amplitude of the initial condition
v_amplitude=0.5

x = range(Δx/2, Lx - Δx/2, length=nx) # cell-centered grid
y = range(Δy/2, Ly - Δy/2, length=ny)

# initial condition
function initial_condition(x, y)
    u0 = [u_mean + u_amplitude * cos(kx * xi) * cos(ky * yi) for yi in y, xi in x]
    v0 = [u_amplitude * sin(kx * xi) * sin(ky * yi) for yi in y, xi in x]
    return u0, v0
end

u0, v0 = initial_condition(x, y)

# added noise strength for stabilizied learning
noise_strength=0.00f0

# time
nt=10000
t_end=15.0
Δt=t_end/nt
println("Using Δt = $Δt, nt = $nt")

Using Δt = 0.0015, nt = 10000


In [3]:
circshift2(u, sx, sy) = circshift(u, (sy, sx))

function convective_flux(u, v)
    nx, ny = size(u)
    dudt = zeros(nx, ny)
    dvdt = zeros(nx, ny)

    u_xp = circshift2(u, -1, 0)
    u_xm = circshift2(u, 1, 0)
    v_yp = circshift2(v, 0, -1)
    v_ym = circshift2(v, 0, 1)

    for j in 1:ny, i in 1:nx
        # x direction
        uL, uR = u[i,j], u_xp[i,j]
        alpha_x = max(abs(uL), abs(uR))
        flux_x_LR = 0.25(uL^2 + uR^2) - 0.5alpha_x*(uR - uL)

        uL_prev, uR_prev = u_xm[i,j], u[i,j]
        alpha_x_prev = max(abs(uL_prev), abs(uR_prev))
        flux_x_prev = 0.25(uL_prev^2 + uR_prev^2) - 0.5alpha_x_prev*(uR_prev - uL_prev)

        # y direction
        vL, vR = v[i,j], v_yp[i,j]
        alpha_y = max(abs(vL), abs(vR))
        flux_y_LR = 0.25(vL^2 + vR^2) - 0.5alpha_y*(vR - vL)

        vL_prev, vR_prev = v_ym[i,j], v[i,j]
        alpha_y_prev = max(abs(vL_prev), abs(vR_prev))
        flux_y_prev = 0.25(vL_prev^2 + vR_prev^2) - 0.5alpha_y_prev*(vR_prev - vL_prev)

        dudt[i,j] = (-1/Δx)*(flux_x_LR - flux_x_prev) + (-1/Δy)*(flux_y_LR - flux_y_prev)
        dvdt[i,j] = (-1/Δx)*(flux_x_LR - flux_x_prev) + (-1/Δy)*(flux_y_LR - flux_y_prev)
    end

    return dudt, dvdt
end

function diffusive_flux(u)
    u_xp = circshift2(u, -1, 0)
    u_xm = circshift2(u, 1, 0)
    u_yp = circshift2(u, 0, -1)
    u_ym = circshift2(u, 0, 1)
    return (ν/Δx^2)*(u_xp - 2u + u_xm) + (ν/Δy^2)*(u_yp - 2u + u_ym)
end

function R(u, v)
    du_conv, dv_conv = convective_flux(u, v)
    du_diff = diffusive_flux(u)
    dv_diff = diffusive_flux(v)
    return du_conv .+ du_diff, dv_conv .+ dv_diff
end

function rk3_step(u, v, Δt)
    k1u, k1v = R(u,v)
    u1 = u .+ Δt .* k1u
    v1 = v .+ Δt .* k1v

    k2u, k2v = R(u1, v1)
    u2 = 0.75u .+ 0.25(u1 .+ Δt .* k2u)
    v2 = 0.75v .+ 0.25(v1 .+ Δt .* k2v)

    k3u, k3v = R(u2, v2)
    u_next = (1/3)*u .+ (2/3)*(u2 .+ Δt .* k3u)
    v_next = (1/3)*v .+ (2/3)*(v2 .+ Δt .* k3v)

    return u_next, v_next
end;

In [4]:
u = copy(u0)
v = copy(v0)
sol_u = [copy(u)]
sol_v = [copy(v)]

for n in 1:nt
    u, v = rk3_step(u, v, Δt)
    u .+= (rand(nx, ny) .- 0.5)*2*noise_strength
    v .+= (rand(nx, ny) .- 0.5)*2*noise_strength
    push!(sol_u, copy(u))
    push!(sol_v, copy(v))
end;

In [5]:
cfl = 0.8
dt_out_target = cfl * min(Δx, Δy) / abs(u_mean)
t_step = Int(floor(dt_out_target / Δt))
dt_out = t_step * Δt
times = collect(0.0:dt_out:t_end)

u_solution = []
v_solution = []
for t_idx in 1:t_step:length(sol_u)
    push!(u_solution, sol_u[t_idx])
    push!(v_solution, sol_v[t_idx])
end

save("data/burgers2d_periodicTEMP.jld2",
    "u_solution", u_solution,
    "v_solution", v_solution,
    "times", times,
    "params", (ν=ν, Δx=Δx, Δy=Δy, u_mean=u_mean, u_amplitude=u_amplitude, kx=kx, ky=ky, Lx=Lx, Ly=Ly),
    "grid_x", x,
    "grid_y", y,
    "cfl_out", cfl,
    "dt_out", dt_out,
    "description", "Solution of the 2D Burgers equation with periodic boundary conditions, using a cosine-sine initial condition."
)